# Анализ программы лояльности

## Задачи проекта:

1. провести исследовательский анализ данных показать общую картину
2. получить основные ритейл-метрики по когортам у клиентов внутри программы лояльности и вне ее
3. проанализировать насколько сработала текущая программа лояльности
4. если программа не слишком эффективна, то возможно, предложить способы повышения эффективности, обосновать использование других программ лояльности
5. если программа достаточно эффективна, то возможно, сказать каких еще клиентов стоит подключить к программе лояльности в первую очередь
6. Сформулировать и проверить гипотезы

## Загрузка библиотек и знакомство с данными

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

In [6]:
product_df = pd.read_csv('data/raw/product_codes.csv')
retail_df = pd.read_csv('data/raw/retail_dataset.csv')

In [64]:
def inform(df):
  print(f"___ Основная информация ___")
  print(f"Память: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
  display(df.head())
  print(f"___ Качество данных ___")
  profile = pd.DataFrame({
    "number_of_rows": df.shape[0],
    "number_of_cols": df.shape[1],
    "duplicates": df.duplicated().sum(),
    "dtype": df.dtypes.astype(str),
    "nunique": df.nunique(dropna=False),
    "gaps": df.isna().sum(),
    "gaps_share": (df.isna().mean() * 100).round(2)
  })
  display(profile.T)
  print(f"___ Описательная статистика ___")
  num = df.select_dtypes(include="number")
  if not num.empty:
    display(num.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2))
    

In [ ]:
# Посмотрим информацию про product_df
inform(product_df)

___ Основная информация ___
Память: 0.6 MB


,productID,price_per_one
0,85123A,2.55
1,71053,3.39
2,84406B,2.75
3,84029G,3.39
4,84029E,3.39


___ Качество данных ___


,productID,price_per_one
number_of_rows,9969,9969
number_of_cols,2,2
duplicates,0,0
dtype,str,float64
nunique,3159,586
gaps,0,0
gaps_share,0.0,0.0


___ Описательная статистика ___


,price_per_one
count,9969.00
mean,19.50
std,330.88
min,0.00
1%,0.00
5%,0.32
25%,1.25
50%,2.55
75%,5.51
95%,16.98


In [67]:
# Посмотрим информацию про retail_df
inform(retail_df)

___ Основная информация ___
Память: 27.0 MB


,purchaseid,item_ID,Quantity,purchasedate,CustomerID,ShopID,loyalty_program
0,538280,21873,11,2016-12-10 12:50:00,18427.0,Shop 0,0.0
1,538862,22195,0,2016-12-14 14:11:00,22389.0,Shop 0,1.0
2,538855,21239,7,2016-12-14 13:50:00,22182.0,Shop 0,1.0
3,543543,22271,0,2017-02-09 15:33:00,23522.0,Shop 0,1.0
4,543812,79321,0,2017-02-13 14:40:00,23151.0,Shop 0,1.0


___ Качество данных ___


,purchaseid,item_ID,Quantity,purchasedate,CustomerID,ShopID,loyalty_program
number_of_rows,105335,105335,105335,105335,105335,105335,105335
number_of_cols,7,7,7,7,7,7,7
duplicates,1033,1033,1033,1033,1033,1033,1033
dtype,str,str,int64,str,float64,str,float64
nunique,4894,3159,301,4430,1750,31,2
gaps,0,0,0,0,36210,0,0
gaps_share,0.0,0.0,0.0,0.0,34.38,0.0,0.0


___ Описательная статистика ___


,Quantity,CustomerID,loyalty_program
count,105335.00,69125.00,105335.00
mean,7.82,21019.30,0.23
std,327.95,1765.44,0.42
min,-74216.00,18025.00,0.00
1%,-3.00,18081.00,0.00
5%,0.00,18273.00,0.00
25%,0.00,19544.00,0.00
50%,2.00,20990.00,0.00
75%,7.00,22659.00,0.00
95%,24.00,23644.00,1.00


## Вывод
Таблица `product_df`:
- Датасет содержит 9969 строк и 2 колонки
- Дубликатов и пропусков нет
- Типы данных соответствуют содержимому признаков
- Минимальная цена равна `0`. Возможно, это товары с нулевой стоимостью или особенности формирования данных. На следующих этапах необходимо отдельно изучить такие значения.
- Максимальная цена составляет `16 888`, что значительно выше 99-го перцентиля `226.63`. Такие значения могут быть потенциальными выбросами, однако на данном этапе преждевременно считать их ошибочными так же необходимо дополнительно проверить их природу.
- Название колонок необходимо привести к единому виду

Таблица `retail_df`:
- Датасет содержит 105335 строк и 7 колонок
- Обнаруженно 1033 дубликатов и 36210 пропусков в колонке `CustomerID` необходимо определить причину этого
- Колонку `purchasedate` необходимо привести к типу `datetime`. А в столбцах `CustomerID` и `loyalty_program` следует поменять на  `int`.
- В столбце `Quantity` есть отрицательные значения, так же необходимо понять причину этого
- Так же название колонок необходимо привести к единому виду